In [1]:

import os
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)



os.makedirs("../outputs", exist_ok=True)



print("Fetching Adult dataset...")

adult = fetch_openml(
    "adult",
    version=2,
    as_frame=True
)

df = adult.frame.copy()

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)



df = df.replace("?", np.nan)

print("Missing values converted to NaN.")


df["target"] = df["class"].map({
    "<=50K": 0,
    ">50K": 1
}).astype(int)


X = df.drop(
    columns=["class", "target"]
)

y = df["target"]


numeric_features = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week"
]

categorical_features = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country"
]


numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)



preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Hold-out test data:", X_test.shape)


logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                random_state=42,
                max_iter=1000
            )
        )
    ]
)


decision_tree_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)



print("\nTraining Logistic Regression...")

logistic_pipeline.fit(
    X_train,
    y_train
)

print("Logistic Regression completed!")


print("\nTraining Decision Tree...")

decision_tree_pipeline.fit(
    X_train,
    y_train
)

print("Decision Tree completed!")


logistic_predictions = logistic_pipeline.predict(
    X_test
)

tree_predictions = decision_tree_pipeline.predict(
    X_test
)


logistic_probabilities = (
    logistic_pipeline
    .predict_proba(X_test)[:, 1]
)

tree_probabilities = (
    decision_tree_pipeline
    .predict_proba(X_test)[:, 1]
)

def evaluate_model(
    y_true,
    y_pred,
    y_probability
):

    return {
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "ROC AUC": roc_auc_score(
            y_true,
            y_probability
        ),

        "PR AUC": average_precision_score(
            y_true,
            y_probability
        )
    }


logistic_metrics = evaluate_model(
    y_test,
    logistic_predictions,
    logistic_probabilities
)

tree_metrics = evaluate_model(
    y_test,
    tree_predictions,
    tree_probabilities
)


comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        **logistic_metrics
    },
    {
        "Model": "Decision Tree",
        **tree_metrics
    }
])


metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC AUC",
    "PR AUC"
]


comparison[metric_columns] = (
    comparison[metric_columns]
    .round(4)
)


print("\n" + "=" * 70)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 70)

print(
    comparison.to_string(index=False)
)


# Save comparison
comparison.to_csv(
    "../outputs/task5_model_comparison.csv",
    index=False
)


print("\n" + "=" * 70)
print("BEST MODEL BY METRIC")
print("=" * 70)

for metric in metric_columns:

    best_index = comparison[metric].idxmax()

    best_model = comparison.loc[
        best_index,
        "Model"
    ]

    best_score = comparison.loc[
        best_index,
        metric
    ]

    print(
        f"{metric}: {best_model} "
        f"({best_score:.4f})"
    )


tree_train_accuracy = (
    decision_tree_pipeline
    .score(
        X_train,
        y_train
    )
)

tree_test_accuracy = (
    decision_tree_pipeline
    .score(
        X_test,
        y_test
    )
)

tree_gap = (
    tree_train_accuracy -
    tree_test_accuracy
)


print("\n" + "=" * 70)
print("DECISION TREE OVERFITTING CHECK")
print("=" * 70)

print(
    "Training Accuracy:",
    round(tree_train_accuracy, 4)
)

print(
    "Test Accuracy:",
    round(tree_test_accuracy, 4)
)

print(
    "Train-Test Gap:",
    round(tree_gap, 4)
)

logistic_f1 = logistic_metrics["F1"]
tree_f1 = tree_metrics["F1"]

logistic_pr_auc = logistic_metrics["PR AUC"]
tree_pr_auc = tree_metrics["PR AUC"]

logistic_roc_auc = logistic_metrics["ROC AUC"]
tree_roc_auc = tree_metrics["ROC AUC"]


logistic_selection_score = (
    0.40 * logistic_f1 +
    0.35 * logistic_pr_auc +
    0.25 * logistic_roc_auc
)

tree_selection_score = (
    0.40 * tree_f1 +
    0.35 * tree_pr_auc +
    0.25 * tree_roc_auc
)


print("\n" + "=" * 70)
print("MODEL SELECTION SCORE")
print("=" * 70)

print(
    "Logistic Regression:",
    round(logistic_selection_score, 4)
)

print(
    "Decision Tree:",
    round(tree_selection_score, 4)
)


if logistic_selection_score >= tree_selection_score:

    selected_model = "Logistic Regression"

else:

    selected_model = "Decision Tree"


print(
    "\nSelected model for further development:",
    selected_model
)


print("\n" + "=" * 70)
print("TASK 5 MODEL SELECTION REPORT")
print("=" * 70)


if selected_model == "Logistic Regression":

    print(
        """
Logistic Regression is selected for further development.

It provides a strong baseline with good classification
performance while remaining highly interpretable. Its
coefficients allow us to understand how numerical and
categorical features influence the prediction of income
greater than $50K.

The model also benefits from standardized numerical features
and one-hot encoded categorical features. Compared with the
Decision Tree, Logistic Regression is generally less prone
to memorizing individual training examples.
"""
    )

else:

    print(
        """
Decision Tree is selected for further development.

The Decision Tree is able to model nonlinear relationships
and feature interactions that may not be captured by a
linear Logistic Regression model. Its rule-based structure
also provides an intuitive way to understand individual
decision paths.

However, its training and test performance should be
carefully monitored because unrestricted Decision Trees
can overfit the training data.
"""
    )


print("\n" + "=" * 70)
print("PREPROCESSING CHANGES TO TEST TOMORROW")
print("=" * 70)

preprocessing_changes = [
    "Test alternative categorical imputation using a constant 'Missing' category.",
    "Compare different Logistic Regression regularization strengths (C values).",
    "Tune Decision Tree max_depth to reduce overfitting.",
    "Test Decision Tree min_samples_split and min_samples_leaf.",
    "Compare model performance with and without feature scaling where appropriate.",
    "Investigate class weighting if recall of the >50K class needs improvement.",
    "Consider threshold tuning to balance precision and recall."
]

for i, change in enumerate(
    preprocessing_changes,
    start=1
):

    print(
        f"{i}. {change}"
    )


report_text = f"""
TASK 5: WRITE-UP & MODEL SELECTION

The Adult income dataset was evaluated using two supervised
learning pipelines: Logistic Regression and Decision Tree.
Both models used the same preprocessing approach consisting
of median imputation and standard scaling for numerical
features, and most-frequent imputation followed by one-hot
encoding for categorical features.

Logistic Regression Accuracy: {logistic_metrics['Accuracy']:.4f}
Logistic Regression Precision: {logistic_metrics['Precision']:.4f}
Logistic Regression Recall: {logistic_metrics['Recall']:.4f}
Logistic Regression F1: {logistic_metrics['F1']:.4f}
Logistic Regression ROC AUC: {logistic_metrics['ROC AUC']:.4f}
Logistic Regression PR AUC: {logistic_metrics['PR AUC']:.4f}

Decision Tree Accuracy: {tree_metrics['Accuracy']:.4f}
Decision Tree Precision: {tree_metrics['Precision']:.4f}
Decision Tree Recall: {tree_metrics['Recall']:.4f}
Decision Tree F1: {tree_metrics['F1']:.4f}
Decision Tree ROC AUC: {tree_metrics['ROC AUC']:.4f}
Decision Tree PR AUC: {tree_metrics['PR AUC']:.4f}

Selected model for further development:
{selected_model}

Decision Tree Training Accuracy: {tree_train_accuracy:.4f}
Decision Tree Test Accuracy: {tree_test_accuracy:.4f}
Decision Tree Train-Test Gap: {tree_gap:.4f}

PREPROCESSING CHANGES PLANNED FOR TOMORROW:

1. Test a constant 'Missing' category for categorical missing values.
2. Test different Logistic Regression regularization strengths.
3. Tune Decision Tree max_depth.
4. Tune min_samples_split and min_samples_leaf.
5. Investigate class weighting.
6. Test probability threshold tuning.
"""


with open(
    "../outputs/task5_model_selection_report.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(report_text)



Fetching Adult dataset...
Dataset loaded successfully!
Dataset shape: (48842, 15)
Missing values converted to NaN.

Training data: (39073, 14)
Hold-out test data: (9769, 14)

Training Logistic Regression...
Logistic Regression completed!

Training Decision Tree...
Decision Tree completed!

MODEL PERFORMANCE COMPARISON
              Model  Accuracy  Precision  Recall     F1  ROC AUC  PR AUC
Logistic Regression    0.8524     0.7411  0.5890 0.6563   0.9042  0.7633
      Decision Tree    0.8141     0.6098  0.6198 0.6148   0.7475  0.4690

BEST MODEL BY METRIC
Accuracy: Logistic Regression (0.8524)
Precision: Logistic Regression (0.7411)
Recall: Decision Tree (0.6198)
F1: Logistic Regression (0.6563)
ROC AUC: Logistic Regression (0.9042)
PR AUC: Logistic Regression (0.7633)

DECISION TREE OVERFITTING CHECK
Training Accuracy: 0.9999
Test Accuracy: 0.8141
Train-Test Gap: 0.1858

MODEL SELECTION SCORE
Logistic Regression: 0.7558
Decision Tree: 0.5969

Selected model for further development: Log